# Почему изменилась выручка ПРАЙМ

Сравниваем два соседних окна по семь завершённых дней UTC на данных твоего сегмента. Модель из С6 умножает число успешных платежей на средний платёж. Сначала меняем число платежей, затем средний платёж.

Второй каскад показывает, в каких каналах привлечения изменилась выручка. По умолчанию выбран отчёт за 23 августа 2026 года для разбора истории ПРАЙМ.

In [ ]:
from pathlib import Path
import sys
from datetime import date, timedelta

import pandas as pd

notebooks_dir = Path.cwd() / "notebooks"
if notebooks_dir.is_dir() and str(notebooks_dir) not in sys.path:
    sys.path.insert(0, str(notebooks_dir))

from prime.analytics_data import load_daily, load_payment_channels

import matplotlib.pyplot as plt

from prime.factors import channel_revenue_factors, weekly_revenue_factors

report_date = date(2026, 8, 23)
week = timedelta(days=7)
daily, _ = load_daily(report_date - 2 * week, report_date)
channels = load_payment_channels(report_date - 2 * week, report_date)
display(daily[["payments_count", "total_revenue", "payments_complete"]])

In [ ]:
factors = weekly_revenue_factors(daily, report_date)
factor_keys = [
    "previous_revenue", "payment_count_effect", "average_payment_effect", "current_revenue",
]
if factors["issues"]:
    print("Причины пропуска разложения:", factors["issues"])
else:
    display(pd.Series({name: factors[name] for name in factor_keys}, name="₽"))

In [ ]:
def show_waterfall(previous: float, current: float, impacts: dict[str, float], title: str):
    labels = ["Прошлая неделя", *impacts, "Последняя неделя"]
    deltas = pd.Series(impacts, dtype=float)
    levels = previous + deltas.cumsum().shift(fill_value=0)
    heights = [previous, *deltas, current]
    bottoms = [0, *levels, 0]
    colors = ["#58708f", *["#238875" if value >= 0 else "#b74b4b" for value in deltas], "#58708f"]

    fig, ax = plt.subplots(figsize=(11, 4.5), layout="constrained")
    bars = ax.bar(labels, heights, bottom=bottoms, color=colors)
    annotations = [f"{previous:,.0f}", *[f"{value:+,.0f}" for value in deltas], f"{current:,.0f}"]
    ax.bar_label(bars, labels=[value.replace(",", " ") for value in annotations], padding=3)
    ax.tick_params(axis="x", labelrotation=20)
    ax.set_ylim(bottom=0)
    ax.margins(y=0.15)
    ax.set_ylabel("₽")
    ax.set_title(title)
    plt.show()


if not factors["issues"]:
    show_waterfall(
        factors["previous_revenue"], factors["current_revenue"],
        {"Число платежей": factors["payment_count_effect"], "Средний платёж": factors["average_payment_effect"]},
        "Изменение выручки за две соседние недели",
    )

Вклад канала — разница выручки между окнами. Перед разложением сверяем выручку и число платежей по каналам с общим итогом каждой недели. Затем разбираем вклад канала, который внёс наибольший вклад в направлении общего изменения.

In [ ]:
by_channel = channel_revenue_factors(channels, factors)
if by_channel["reason"]:
    print("Причина пропуска разбивки:", by_channel["reason"])
else:
    contributions = by_channel["channels"]
    if contributions:
        display(pd.DataFrame(contributions))
        show_waterfall(
            factors["previous_revenue"], factors["current_revenue"],
            {row["channel"]: row["delta"] for row in contributions},
            "Вклад каналов привлечения в изменение выручки",
        )
    if by_channel["leading"]:
        display(pd.Series(by_channel["leading"], name="Ведущий канал"))